# 04 — Trigger Batch Inference Pipeline

This notebook runs the **SageMaker Batch Transform** pipeline in local mode.

**Prerequisites:** The training pipeline (notebook 03) must have completed successfully,
so that a champion model exists in MLflow and the preprocessing artifacts are on S3.

### Pipeline Steps

1. **Build** the inference Docker image (`credit-risk-inference:latest`)
2. **Detect** the latest ingestion date for test features in the gold layer
3. **Step 1 — Prepare Model:** Downloads champion model + preprocessor from MLflow, packages `model.tar.gz`
4. **Step 2 — Batch Transform:** Runs the inference container against gold test features and writes predictions to S3
5. **Verify** the predictions output on S3


## 1. Install Dependencies


In [1]:
!uv pip install -q "sagemaker>=2.0.0,<3.0.0" "sagemaker[local]" "boto3==1.42.82" "pytz==2026.1.post1" "requests==2.33.1" "pyarrow==23.0.1"

In [2]:
!uv pip freeze | grep -E "sagemaker|boto3|pytz|requests|pyarrow"

boto3==1.42.82
pyarrow==23.0.1
pytz==2026.1.post1
requests==2.33.1
sagemaker==2.257.1
sagemaker-core==1.0.77


In [3]:
import os
import subprocess
import sys
from pprint import pprint

import boto3
import pandas as pd

## 2. Build the Inference Docker Image


In [4]:
# !docker build --no-cache -q -t credit-risk-inference:latest -f /app/give_me_some_credit/sagemaker/batch_inference/Dockerfile /app/give_me_some_credit/sagemaker/batch_inference/

## 3. Inspect S3 — Gold Test Features

Verify the test data (inference data) exists in the gold layer before running the pipeline.


In [5]:
!aws s3 ls s3://data-lake/gold/give_me_some_credit/test_features/ --recursive

2026-04-04 21:08:39    1329743 gold/give_me_some_credit/test_features/ingestion_date=2026-04-04/part-00000-a10543aa-4ecc-43e2-aefa-2f14a3f8acc6.c000.snappy.parquet
2026-04-04 21:08:39    1322762 gold/give_me_some_credit/test_features/ingestion_date=2026-04-04/part-00001-a10543aa-4ecc-43e2-aefa-2f14a3f8acc6.c000.snappy.parquet


## 4. Detect Latest Ingestion Date

Scan the S3 gold test features partitions and pick the most recent date.


In [6]:
def get_latest_ingestion_date(bucket, prefix, s3_endpoint=None):
    s3 = boto3.client("s3", endpoint_url=s3_endpoint)

    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=bucket, Prefix=prefix, Delimiter="/")

    dates = []
    for page in pages:
        for obj in page.get("CommonPrefixes", []):
            folder_name = obj.get("Prefix").split("=")[-1].strip("/")
            dates.append(folder_name)

    if not dates:
        raise ValueError(f"No partitions found in s3://{bucket}/{prefix}")

    return sorted(dates)[-1]


S3_ENDPOINT = "http://localstack:4566"
LATEST_DATE = get_latest_ingestion_date(
    bucket="data-lake",
    prefix="gold/give_me_some_credit/test_features/",
    s3_endpoint=S3_ENDPOINT,
)

print(f"Detected Latest Ingestion Date: {LATEST_DATE}")

Detected Latest Ingestion Date: 2026-04-04


## 5. Inspect Training Artifacts on S3

Verify the training pipeline left the required artifacts (evaluation report, preprocessor metadata) that the batch inference pipeline needs.


In [7]:
EXPERIMENT_NAME = "give_me_some_credit"
PIPELINE_S3_PREFIX = f"projects/{EXPERIMENT_NAME}/sagemaker/pipeline"

print("=== Evaluation artifacts ===")
!aws s3 ls s3://data-lake/{PIPELINE_S3_PREFIX}/evaluation/ --recursive

print("\n=== Preprocessor metadata ===")
!aws s3 ls s3://data-lake/{PIPELINE_S3_PREFIX}/preprocessing/preprocessor/ --recursive

=== Evaluation artifacts ===


2026-04-04 23:01:55        916 projects/give_me_some_credit/sagemaker/pipeline/evaluation/evaluation_report.json



=== Preprocessor metadata ===


2026-04-04 22:59:31       1357 projects/give_me_some_credit/sagemaker/pipeline/preprocessing/preprocessor/feature_config.json
2026-04-04 22:59:31         46 projects/give_me_some_credit/sagemaker/pipeline/preprocessing/preprocessor/prep_meta.json


## 6. Run the Batch Inference Pipeline

This triggers `sm_batch_inference.py` in local mode. It will:

1. **Prepare Model** — pull champion model + preprocessor from MLflow, create `model.tar.gz`
2. **Batch Transform** — run the inference container on the gold test features


In [8]:
NETWORK = os.environ["NETWORK"]
result = subprocess.run(
    [
        sys.executable,
        "/app/give_me_some_credit/sagemaker/batch_inference/sm_batch_inference.py",
        "--mode",
        "local",
        "--ingestion-date",
        LATEST_DATE,
        "--s3-endpoint",
        "http://localstack:4566",
        "--network",
        NETWORK,
    ],
    capture_output=False,
    text=True,
)
print(f"\nExit code: {result.returncode}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


2026-04-04 23:03:40,772 - sagemaker_batch_inference - INFO - Args: {'mode': 'local', 'ingestion_date': '2026-04-04', 's3_endpoint': 'http://localstack:4566', 's3_bucket': 'data-lake', 'mlflow_uri': 'http://mlflow:5000', 'experiment_name': 'give_me_some_credit', 'training_image': 'credit-risk-training:latest', 'inference_image': 'credit-risk-inference:latest', 'network': 'mlops-lab-net', 'aws_region': 'us-east-1'}
2026-04-04 23:03:40,830 - botocore.configprovider - INFO - Found endpoint for s3 via: environment_global.
2026-04-04 23:03:40,834 - botocore.configprovider - INFO - Found endpoint for s3 via: environment_global.
2026-04-04 23:03:40,843 - sagemaker_session - INFO - Local SageMaker session ready (bucket=data-lake)
2026-04-04 23:03:40,843 - sagemaker_batch_inference - INFO - Step 1: Preparing model artifacts...
2026-04-04 23:03:40,898 - sagemaker - INFO - Creating processing-job with name credit-risk-training-2026-04-04-23-03-40-843
2026-04-04 23:03:40,899 - sagemaker.telemetry.t

2026-04-04 23:03:41,013 - sagemaker.local.image - INFO - 'Docker Compose' found using Docker Compose CLI.
2026-04-04 23:03:41,013 - sagemaker.local.local_session - INFO - Starting processing job
2026-04-04 23:03:41,040 - sagemaker.local.image - INFO - Using the long-lived AWS credentials found in session
2026-04-04 23:03:41,042 - sagemaker.local.image - INFO - docker compose file: 
networks:
  sagemaker-local:
    name: sagemaker-local
services:
  algo-1-w4kpx:
    container_name: fry7uszaj0-algo-1-w4kpx
    entrypoint:
    - opentelemetry-instrument
    - python
    - /opt/ml/processing/input/code/prepare_model.py
    - --mlflow-uri
    - http://mlflow:5000
    - --experiment-name
    - give_me_some_credit
    environment:
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Mas

2026-04-04 23:03:49,577 - sagemaker.local.image - INFO - ===== Job Complete =====
2026-04-04 23:03:49,578 - botocore.configprovider - INFO - Found endpoint for sts via: environment_global.


2026-04-04 23:03:50,215 - sagemaker_batch_inference - INFO - Step 1 complete: model.tar.gz uploaded to S3.
2026-04-04 23:03:50,216 - sagemaker_batch_inference - INFO - Step 2: Running Batch Transform...
2026-04-04 23:03:50,217 - sagemaker - INFO - Creating model with name: credit-risk-inference-2026-04-04-23-03-50-216
2026-04-04 23:03:50,217 - sagemaker.telemetry.telemetry_logging - INFO - SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
2026-04-04 23:03:50,218 - botocore.configprovider - INFO - Found endpoint for sts via: environment_global.


...

2026-04-04 23:03:58,407 - sagemaker_session - INFO - Resolved Docker host gateway: 172.18.0.1


2026-04-04 23:03:59,440 - botocore.configprovider - INFO - Found endpoint for sts via: environment_global.


2026-04-04 23:04:09,959 - botocore.configprovider - INFO - Found endpoint for logs via: environment_global.
2026-04-04 23:04:09,969 - sagemaker_batch_inference - INFO - Step 2 complete: predictions written to s3://data-lake/projects/give_me_some_credit/sagemaker/pipeline/inference/predictions/ingestion_date=2026-04-04
2026-04-04 23:04:09,969 - sagemaker_batch_inference - INFO - Batch inference pipeline complete.


time="2026-04-04T23:03:41Z" level=warning msg="/tmp/tmprt02afcn/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-04-04T23:03:41Z" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmprt02afcn\".\nSet `external: true` to use an existing network"
 Container fry7uszaj0-algo-1-w4kpx Creating 
 Container fry7uszaj0-algo-1-w4kpx Created 
Attaching to fry7uszaj0-algo-1-w4kpx
 Container fry7uszaj0-algo-1-w4kpx Starting 
 Container fry7uszaj0-algo-1-w4kpx Started 
fry7uszaj0-algo-1-w4kpx  | 2026-04-04 23:03:48,193 - prepare_model - INFO - Args: {'mlflow_uri': 'http://mlflow:5000', 'experiment_name': 'give_me_some_credit'}
fry7uszaj0-algo-1-w4kpx  | 2026-04-04 23:03:48,213 - prepare_model - INFO - Found give_me_some_credit_champion v4 (alias=champion, run_id=4a39499400444a5eb748417451c2378e)
fry7uszaj0-algo-1-w4kpx  | 2026-04-04 23:03:48,213 - prepare_model - INFO 


Exit code: 0


## 7. Verify Predictions on S3

Check that the predictions were written to the expected location.


In [9]:
PREDICTIONS_PREFIX = (
    f"{PIPELINE_S3_PREFIX}/inference/predictions/ingestion_date={LATEST_DATE}"
)
print(f"Looking for predictions at: s3://data-lake/{PREDICTIONS_PREFIX}/")
!aws s3 ls s3://data-lake/{PREDICTIONS_PREFIX}/ --recursive

Looking for predictions at: s3://data-lake/projects/give_me_some_credit/sagemaker/pipeline/inference/predictions/ingestion_date=2026-04-04/


2026-04-04 22:04:46     604297 projects/give_me_some_credit/sagemaker/pipeline/inference/predictions/ingestion_date=2026-04-04/credit-risk-inference-2026-04-04-22-04-40-663/part-00000-a10543aa-4ecc-43e2-aefa-2f14a3f8acc6.c000.snappy.parquet.out
2026-04-04 22:04:46     602558 projects/give_me_some_credit/sagemaker/pipeline/inference/predictions/ingestion_date=2026-04-04/credit-risk-inference-2026-04-04-22-04-40-663/part-00001-a10543aa-4ecc-43e2-aefa-2f14a3f8acc6.c000.snappy.parquet.out
2026-04-04 22:16:04     604297 projects/give_me_some_credit/sagemaker/pipeline/inference/predictions/ingestion_date=2026-04-04/credit-risk-inference-2026-04-04-22-15-58-014/part-00000-a10543aa-4ecc-43e2-aefa-2f14a3f8acc6.c000.snappy.parquet.out
2026-04-04 22:16:04     602558 projects/give_me_some_credit/sagemaker/pipeline/inference/predictions/ingestion_date=2026-04-04/credit-risk-inference-2026-04-04-22-15-58-014/part-00001-a10543aa-4ecc-43e2-aefa-2f14a3f8acc6.c000.snappy.parquet.out
2026-04-04 22:19:33 

## 8. Load & Inspect Predictions

Download the predictions parquet file and display a sample + summary statistics.


In [10]:
import io

s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT)

# List all prediction files
paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(Bucket="data-lake", Prefix=PREDICTIONS_PREFIX)

prediction_files = []
for page in pages:
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet") or obj["Key"].endswith(".out"):
            prediction_files.append(obj["Key"])
            print(f"  {obj['Key']}  ({obj['Size']:,} bytes)")

print(f"\nFound {len(prediction_files)} prediction file(s)")

  projects/give_me_some_credit/sagemaker/pipeline/inference/predictions/ingestion_date=2026-04-04/credit-risk-inference-2026-04-04-22-04-40-663/part-00000-a10543aa-4ecc-43e2-aefa-2f14a3f8acc6.c000.snappy.parquet.out  (604,297 bytes)
  projects/give_me_some_credit/sagemaker/pipeline/inference/predictions/ingestion_date=2026-04-04/credit-risk-inference-2026-04-04-22-04-40-663/part-00001-a10543aa-4ecc-43e2-aefa-2f14a3f8acc6.c000.snappy.parquet.out  (602,558 bytes)
  projects/give_me_some_credit/sagemaker/pipeline/inference/predictions/ingestion_date=2026-04-04/credit-risk-inference-2026-04-04-22-15-58-014/part-00000-a10543aa-4ecc-43e2-aefa-2f14a3f8acc6.c000.snappy.parquet.out  (604,297 bytes)
  projects/give_me_some_credit/sagemaker/pipeline/inference/predictions/ingestion_date=2026-04-04/credit-risk-inference-2026-04-04-22-15-58-014/part-00001-a10543aa-4ecc-43e2-aefa-2f14a3f8acc6.c000.snappy.parquet.out  (602,558 bytes)
  projects/give_me_some_credit/sagemaker/pipeline/inference/predicti

In [11]:
# Read the first prediction file and inspect results
if prediction_files:
    obj = s3.get_object(Bucket="data-lake", Key=prediction_files[0])
    df_preds = pd.read_parquet(io.BytesIO(obj["Body"].read()))

    print(f"Shape: {df_preds.shape}")
    print(f"\nColumns: {list(df_preds.columns)}")
    print(f"\n--- First 10 rows ---")
    display(df_preds.head(10))

    print(f"\n--- Prediction distribution ---")
    print(df_preds["prediction"].value_counts())

    print(f"\n--- Probability statistics ---")
    print(df_preds["probability"].describe())
else:
    print("No prediction files found — check pipeline logs above.")

Shape: (50669, 3)

Columns: ['row_id', 'probability', 'prediction']

--- First 10 rows ---


,row_id,probability,prediction
0,0,0.509509,0
1,1,0.407834,0
2,2,0.517210,0
3,3,0.286652,0
4,4,0.291798,0
5,5,0.332531,0
6,6,0.920650,1
7,7,0.093934,0
8,8,0.087580,0
9,9,0.451020,0



--- Prediction distribution ---
prediction
0    41454
1     9215
Name: count, dtype: int64

--- Probability statistics ---
count    50669.000000
mean         0.320636
std          0.260432
min          0.018537
25%          0.106331
50%          0.221704
75%          0.493277
max          0.985798
Name: probability, dtype: float64


## 9. Inspect Model Artifacts

Verify the `model.tar.gz` that was created by the prepare_model step.


In [12]:
MODEL_S3_KEY = f"{PIPELINE_S3_PREFIX}/inference/model/model.tar.gz"

print(f"Model artifact: s3://data-lake/{MODEL_S3_KEY}")
!aws s3 ls s3://data-lake/{MODEL_S3_KEY}

# Download and list contents of model.tar.gz
import tarfile
import tempfile

with tempfile.NamedTemporaryFile(suffix=".tar.gz") as tmp:
    s3.download_fileobj("data-lake", MODEL_S3_KEY, tmp)
    tmp.seek(0)

    with tarfile.open(tmp.name, "r:gz") as tar:
        print("\nmodel.tar.gz contents:")
        for member in tar.getmembers():
            print(f"  {member.name}  ({member.size:,} bytes)")

Model artifact: s3://data-lake/projects/give_me_some_credit/sagemaker/pipeline/inference/model/model.tar.gz


2026-04-04 23:03:49      54081 model.tar.gz



model.tar.gz contents:
  evaluation_report.json  (916 bytes)
  champion  (0 bytes)
  champion/MLmodel  (543 bytes)
  champion/conda.yaml  (229 bytes)
  champion/model.pkl  (280,666 bytes)
  champion/python_env.yaml  (121 bytes)
  champion/requirements.txt  (109 bytes)
  preprocessor  (0 bytes)
  preprocessor/MLmodel  (350 bytes)
  preprocessor/conda.yaml  (210 bytes)
  preprocessor/model.pkl  (3,693 bytes)
  preprocessor/python_env.yaml  (121 bytes)
  preprocessor/requirements.txt  (94 bytes)
